# CMGAN Training Notebook

Notebook version of the training logic from train.py.

This version also includes Learning without Forgetting (LwF) with configurable $\lambda_{LwF}$ and `n_reset` for the snapshot interval.

In [2]:
%pip install torchinfo natsort einops

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [einops]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
import csv
import logging
import os
from collections import OrderedDict

import torch
import torch.nn.functional as F
import torchaudio
from torch.utils.data import DataLoader
from torchinfo import summary

from data import dataloader
from models import discriminator
from models.generator import TSCNet
from utils import power_compress, power_uncompress

logging.basicConfig(level=logging.INFO)
if torch.cuda.is_available():
    torch.backends.cudnn.benchmark = True

ModuleNotFoundError: No module named 'torch'

In [ ]:
epochs = 120
batch_size = 4
log_interval = 500
decay_epoch = 30
init_lr = 5e-4
cut_len = 16000 * 2
data_dir = "/home/lugra002/speechbrain/data/cmgan_voicebank"
save_model_dir = "./saved_model"
metrics_csv_name = "train_metrics.csv"
loss_weights = [0.1, 0.9, 0.2, 0.05]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_workers = 2

# LwF settings (Li & Hoiem, 2017)
lwf_enabled = True
lambda_lwf = 1.0
lambda_lwf_decay = 1.0
# Optional linear schedule override (takes precedence over lambda_lwf/lambda_lwf_decay).
lambda_lwf_start = None
lambda_lwf_end = None
n_reset = 10
kd_temperature = 2.0

def load_data_notebook(ds_dir, batch_size, num_workers, cut_len):
    train_dir = os.path.join(ds_dir, "train")
    test_dir = os.path.join(ds_dir, "test")

    train_ds = dataloader.DemandDataset(train_dir, cut_len)
    test_ds = dataloader.DemandDataset(test_dir, cut_len)

    train_loader = DataLoader(
        dataset=train_ds,
        batch_size=batch_size,
        pin_memory=torch.cuda.is_available(),
        shuffle=True,
        drop_last=True,
        num_workers=num_workers,
    )
    test_loader = DataLoader(
        dataset=test_ds,
        batch_size=batch_size,
        pin_memory=torch.cuda.is_available(),
        shuffle=False,
        drop_last=False,
        num_workers=num_workers,
    )

    return train_loader, test_loader

NameError: name 'torch' is not defined

In [ ]:
import os

# LwF ablations: vary the weight and the snapshot interval.
ABLATION_PRESETS = {
    "lwf_l0p5_n50": {
        "lwf_enabled": True,
        "lambda_lwf": 0.5,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 50,
    },
    "lwf_l0p5_n10": {
        "lwf_enabled": True,
        "lambda_lwf": 0.5,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 10,
    },
    "lwf_l0p7_n10": {
        "lwf_enabled": True,
        "lambda_lwf": 0.7,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 10,
    },
    "lwf_l0p7_n50": {
        "lwf_enabled": True,
        "lambda_lwf": 0.7,
        "lambda_lwf_decay": 1.0,
        "lambda_lwf_start": None,
        "lambda_lwf_end": None,
        "n_reset": 50,
    },
}

def apply_ablation_preset(name, results_root="./saved_model_ablations"):
    global lwf_enabled, lambda_lwf, lambda_lwf_decay, lambda_lwf_start, lambda_lwf_end, n_reset
    global save_model_dir, metrics_csv_name

    if name not in ABLATION_PRESETS:
        raise ValueError(f"Unknown preset '{name}'. Available: {list(ABLATION_PRESETS.keys())}")

    cfg = ABLATION_PRESETS[name]
    lwf_enabled = cfg["lwf_enabled"]
    lambda_lwf = cfg["lambda_lwf"]
    lambda_lwf_decay = cfg["lambda_lwf_decay"]
    lambda_lwf_start = cfg["lambda_lwf_start"]
    lambda_lwf_end = cfg["lambda_lwf_end"]
    n_reset = cfg["n_reset"]

    save_model_dir = os.path.join(results_root, name)
    metrics_csv_name = "train_metrics.csv"

    print("Applied ablation:")
    print(f"  preset          : {name}")
    print(f"  lwf_enabled     : {lwf_enabled}")
    print(f"  lambda_lwf      : {lambda_lwf}")
    print(f"  lambda_lwf_decay: {lambda_lwf_decay}")
    print(f"  lambda_start    : {lambda_lwf_start}")
    print(f"  lambda_end      : {lambda_lwf_end}")
    print(f"  n_reset         : {n_reset}")
    print(f"  save_model_dir  : {save_model_dir}")

print("Available presets:")
for preset_name in ABLATION_PRESETS:
    print(" -", preset_name)

Available presets:
 - baseline_no_lwf
 - lwf_l0p5_n10
 - lwf_l1p0_n10
 - lwf_l1p0_n50


## Ablation Presets

- `lwf_l0p5_n10`: LwF with lambda = 0.5 and `n_reset = 10`
- `lwf_l0p7_n10`: LwF with lambda = 0.7 and `n_reset = 10`
- `lwf_l0p5_n50`: LwF with lambda = 0.5 and `n_reset = 50`
- `lwf_l0p7_n50`: LwF with lambda = 0.7 and `n_reset = 50`

In [ ]:
import copy

class Trainer:
    def __init__(self, train_ds, test_ds, gpu_id=None):
        self.n_fft = 400
        self.hop = 100
        self.train_ds = train_ds
        self.test_ds = test_ds
        self.gpu_id = gpu_id
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = TSCNet(num_channel=64, num_features=self.n_fft // 2 + 1).to(self.device)
        summary(
            self.model, [(1, 2, cut_len // self.hop + 1, int(self.n_fft / 2) + 1)]
        )
        self.discriminator = discriminator.Discriminator(ndf=16).to(self.device)
        summary(
            self.discriminator,
            [
                (1, 1, int(self.n_fft / 2) + 1, cut_len // self.hop + 1),
                (1, 1, int(self.n_fft / 2) + 1, cut_len // self.hop + 1),
            ],
        )
        self.optimizer = torch.optim.AdamW(self.model.parameters(), lr=init_lr)
        self.optimizer_disc = torch.optim.AdamW(
            self.discriminator.parameters(), lr=2 * init_lr
        )
        self.metrics_csv_path = os.path.join(save_model_dir, metrics_csv_name)
        self.old_generator_model = None
        self.old_discriminator_model = None

    def _maybe_to_device(self, tensor):
        return tensor.to(self.device)

    def _forward_generator_step_with_model(self, model, clean, noisy):
        c = torch.sqrt(noisy.size(-1) / torch.sum((noisy**2.0), dim=-1))
        noisy, clean = torch.transpose(noisy, 0, 1), torch.transpose(clean, 0, 1)
        noisy, clean = torch.transpose(noisy * c, 0, 1), torch.transpose(clean * c, 0, 1)

        window = torch.hamming_window(self.n_fft, device=self.device)
        noisy_spec = torch.stft(
            noisy,
            self.n_fft,
            self.hop,
            window=window,
            onesided=True,
        )
        clean_spec = torch.stft(
            clean,
            self.n_fft,
            self.hop,
            window=window,
            onesided=True,
        )
        noisy_spec = power_compress(noisy_spec).permute(0, 1, 3, 2)
        clean_spec = power_compress(clean_spec)
        clean_real = clean_spec[:, 0, :, :].unsqueeze(1)
        clean_imag = clean_spec[:, 1, :, :].unsqueeze(1)

        est_real, est_imag = model(noisy_spec)
        est_real, est_imag = est_real.permute(0, 1, 3, 2), est_imag.permute(0, 1, 3, 2)
        est_mag = torch.sqrt(est_real**2 + est_imag**2)
        clean_mag = torch.sqrt(clean_real**2 + clean_imag**2)

        est_spec_uncompress = power_uncompress(est_real, est_imag).squeeze(1)
        est_audio = torch.istft(
            est_spec_uncompress,
            self.n_fft,
            self.hop,
            window=window,
            onesided=True,
        )

        return {
            "est_real": est_real,
            "est_imag": est_imag,
            "est_mag": est_mag,
            "clean_real": clean_real,
            "clean_imag": clean_imag,
            "clean_mag": clean_mag,
            "est_audio": est_audio,
        }

    def forward_generator_step(self, clean, noisy):
        return self._forward_generator_step_with_model(self.model, clean, noisy)

    def current_lambda_lwf(self, epoch):
        if not lwf_enabled:
            return 0.0
        if lambda_lwf_start is not None or lambda_lwf_end is not None:
            start_val = float(lambda_lwf_start) if lambda_lwf_start is not None else 0.0
            end_val = float(lambda_lwf_end) if lambda_lwf_end is not None else start_val
            total_epochs = max(int(epochs), 1)
            progress = 0.0 if total_epochs == 1 else min(max(epoch, 0), total_epochs - 1) / (total_epochs - 1)
            return max(0.0, start_val + progress * (end_val - start_val))
        decay = float(lambda_lwf_decay)
        return max(0.0, float(lambda_lwf) * (decay ** epoch))

    def maybe_refresh_lwf_snapshots(self, epoch):
        if not lwf_enabled:
            self.old_generator_model = None
            self.old_discriminator_model = None
            return

        refresh_needed = (
            self.old_generator_model is None
            or self.old_discriminator_model is None
            or max(int(n_reset), 1) > 0 and epoch % max(int(n_reset), 1) == 0
        )
        if refresh_needed:
            self.old_generator_model = copy.deepcopy(self.model).to(self.device)
            self.old_discriminator_model = copy.deepcopy(self.discriminator).to(self.device)
            self.old_generator_model.eval()
            self.old_discriminator_model.eval()
            for module in (self.old_generator_model, self.old_discriminator_model):
                for parameter in module.parameters():
                    parameter.requires_grad_(False)

    def calculate_generator_loss(self, generator_outputs, lwf_loss=0.0):
        predict_fake_metric = self.discriminator(
            generator_outputs["clean_mag"], generator_outputs["est_mag"]
        )
        gen_loss_gan = F.mse_loss(
            predict_fake_metric.flatten(), generator_outputs["one_labels"].float()
        )
        loss_mag = F.mse_loss(
            generator_outputs["est_mag"], generator_outputs["clean_mag"]
        )
        loss_ri = F.mse_loss(
            generator_outputs["est_real"], generator_outputs["clean_real"]
        ) + F.mse_loss(generator_outputs["est_imag"], generator_outputs["clean_imag"])
        time_loss = torch.mean(
            torch.abs(generator_outputs["est_audio"] - generator_outputs["clean"])
        )
        loss = (
            loss_weights[0] * loss_ri
            + loss_weights[1] * loss_mag
            + loss_weights[2] * time_loss
            + loss_weights[3] * gen_loss_gan
            + lwf_loss
        )
        return loss, {
            "loss_ri": loss_ri,
            "loss_mag": loss_mag,
            "time_loss": time_loss,
            "gen_loss_gan": gen_loss_gan,
        }

    def calculate_discriminator_loss(self, generator_outputs, compute_pesq=False, lwf_loss=0.0):
        length = generator_outputs["est_audio"].size(-1)
        pesq_score = None
        if compute_pesq:
            est_audio_list = list(generator_outputs["est_audio"].detach().cpu().numpy())
            clean_audio_list = list(generator_outputs["clean"].cpu().numpy()[:, :length])
            pesq_score = discriminator.batch_pesq(clean_audio_list, est_audio_list)

        if pesq_score is not None:
            predict_enhance_metric = self.discriminator(
                generator_outputs["clean_mag"], generator_outputs["est_mag"].detach()
            )
            predict_max_metric = self.discriminator(
                generator_outputs["clean_mag"], generator_outputs["clean_mag"]
            )
            discrim_loss_metric = F.mse_loss(
                predict_max_metric.flatten(), generator_outputs["one_labels"]
            ) + F.mse_loss(predict_enhance_metric.flatten(), pesq_score)
        else:
            discrim_loss_metric = None

        if discrim_loss_metric is None:
            discrim_loss_metric = torch.tensor(0.0, device=self.device)

        return discrim_loss_metric + lwf_loss, pesq_score

    def train_step(self, batch, epoch):
        clean = self._maybe_to_device(batch[0])
        noisy = self._maybe_to_device(batch[1])
        one_labels = torch.ones(clean.size(0), device=self.device)

        generator_outputs = self.forward_generator_step(clean, noisy)
        generator_outputs["one_labels"] = one_labels
        generator_outputs["clean"] = clean

        lwf_lambda_value = self.current_lambda_lwf(epoch)
        lwf_gen_loss = torch.tensor(0.0, device=self.device)
        lwf_disc_loss = torch.tensor(0.0, device=self.device)
        if lwf_lambda_value > 0.0 and self.old_generator_model is not None:
            with torch.no_grad():
                old_outputs = self._forward_generator_step_with_model(
                    self.old_generator_model, clean, noisy
                )
                old_fake_metric = self.old_discriminator_model(
                    old_outputs["clean_mag"], old_outputs["est_mag"]
                )
            lwf_gen_loss = F.mse_loss(generator_outputs["est_mag"], old_outputs["est_mag"].detach())
            current_fake_metric = self.discriminator(
                generator_outputs["clean_mag"], generator_outputs["est_mag"].detach()
            )
            lwf_disc_loss = F.mse_loss(current_fake_metric, old_fake_metric.detach())

        gen_loss, gen_parts = self.calculate_generator_loss(
            generator_outputs, lwf_loss=lwf_lambda_value * lwf_gen_loss
        )
        self.optimizer.zero_grad()
        gen_loss.backward()
        self.optimizer.step()

        disc_loss, pesq_score = self.calculate_discriminator_loss(
            generator_outputs,
            compute_pesq=False,
            lwf_loss=lwf_lambda_value * lwf_disc_loss,
        )
        self.optimizer_disc.zero_grad()
        disc_loss.backward()
        self.optimizer_disc.step()

        return {
            "gen_loss": float(gen_loss.item()),
            "disc_loss": float(disc_loss.item()),
            "loss_ri": float(gen_parts["loss_ri"].item()),
            "loss_mag": float(gen_parts["loss_mag"].item()),
            "time_loss": float(gen_parts["time_loss"].item()),
            "gen_loss_gan": float(gen_parts["gen_loss_gan"].item()),
            "lwf_gen_loss": float(lwf_gen_loss.item()),
            "lwf_disc_loss": float(lwf_disc_loss.item()),
            "lwf_lambda": float(lwf_lambda_value),
            "pesq_mean": None if pesq_score is None else float(pesq_score),
        }

    @torch.no_grad()
    def test_step(self, batch, compute_pesq=False):
        clean = self._maybe_to_device(batch[0])
        noisy = self._maybe_to_device(batch[1])
        one_labels = torch.ones(clean.size(0), device=self.device)

        generator_outputs = self.forward_generator_step(clean, noisy)
        generator_outputs["one_labels"] = one_labels
        generator_outputs["clean"] = clean

        lwf_lambda_value = self.current_lambda_lwf(epochs - 1) if self.old_generator_model is not None else 0.0
        lwf_gen_loss = torch.tensor(0.0, device=self.device)
        lwf_disc_loss = torch.tensor(0.0, device=self.device)
        if lwf_lambda_value > 0.0 and self.old_generator_model is not None:
            old_outputs = self._forward_generator_step_with_model(
                self.old_generator_model, clean, noisy
            )
            old_fake_metric = self.old_discriminator_model(
                old_outputs["clean_mag"], old_outputs["est_mag"]
            )
            lwf_gen_loss = F.mse_loss(generator_outputs["est_mag"], old_outputs["est_mag"])
            current_fake_metric = self.discriminator(
                generator_outputs["clean_mag"], generator_outputs["est_mag"]
            )
            lwf_disc_loss = F.mse_loss(current_fake_metric, old_fake_metric)

        gen_loss, gen_parts = self.calculate_generator_loss(
            generator_outputs, lwf_loss=lwf_lambda_value * lwf_gen_loss
        )
        disc_loss, pesq_score = self.calculate_discriminator_loss(
            generator_outputs,
            compute_pesq=compute_pesq,
            lwf_loss=lwf_lambda_value * lwf_disc_loss,
        )
        return {
            "gen_loss": float(gen_loss.item()),
            "disc_loss": float(disc_loss.item()),
            "loss_ri": float(gen_parts["loss_ri"].item()),
            "loss_mag": float(gen_parts["loss_mag"].item()),
            "time_loss": float(gen_parts["time_loss"].item()),
            "gen_loss_gan": float(gen_parts["gen_loss_gan"].item()),
            "lwf_gen_loss": float(lwf_gen_loss.item()),
            "lwf_disc_loss": float(lwf_disc_loss.item()),
            "lwf_lambda": float(lwf_lambda_value),
            "pesq_mean": None if pesq_score is None else float(pesq_score),
        }

    def _init_epoch_tracker(self):
        return {
            "steps": 0,
            "gen_loss": 0.0,
            "disc_loss": 0.0,
            "loss_ri": 0.0,
            "loss_mag": 0.0,
            "time_loss": 0.0,
            "gen_loss_gan": 0.0,
            "lwf_gen_loss": 0.0,
            "lwf_disc_loss": 0.0,
            "lwf_lambda_sum": 0.0,
            "pesq_sum": 0.0,
            "pesq_count": 0,
        }

    def _update_epoch_tracker(self, tracker, step_stats):
        tracker["steps"] += 1
        tracker["gen_loss"] += step_stats["gen_loss"]
        tracker["disc_loss"] += step_stats["disc_loss"]
        tracker["loss_ri"] += step_stats["loss_ri"]
        tracker["loss_mag"] += step_stats["loss_mag"]
        tracker["time_loss"] += step_stats["time_loss"]
        tracker["gen_loss_gan"] += step_stats["gen_loss_gan"]
        tracker["lwf_gen_loss"] += step_stats["lwf_gen_loss"]
        tracker["lwf_disc_loss"] += step_stats["lwf_disc_loss"]
        tracker["lwf_lambda_sum"] += step_stats.get("lwf_lambda", 0.0)

        if step_stats["pesq_mean"] is not None:
            tracker["pesq_sum"] += step_stats["pesq_mean"]
            tracker["pesq_count"] += 1

    def _finalize_epoch_tracker(self, tracker):
        steps = max(tracker["steps"], 1)
        stats = {
            "gen_loss": tracker["gen_loss"] / steps,
            "disc_loss": tracker["disc_loss"] / steps,
            "loss_ri": tracker["loss_ri"] / steps,
            "loss_mag": tracker["loss_mag"] / steps,
            "time_loss": tracker["time_loss"] / steps,
            "gen_loss_gan": tracker["gen_loss_gan"] / steps,
            "lwf_gen_loss": tracker["lwf_gen_loss"] / steps,
            "lwf_disc_loss": tracker["lwf_disc_loss"] / steps,
            "lwf_lambda": tracker["lwf_lambda_sum"] / steps,
            "pesq_mean": None,
        }
        if tracker["pesq_count"] > 0:
            stats["pesq_mean"] = tracker["pesq_sum"] / tracker["pesq_count"]
        return stats

    def train(self, start_epoch=0):
        scheduler_G = torch.optim.lr_scheduler.StepLR(
            self.optimizer, step_size=decay_epoch, gamma=0.5
        )
        scheduler_D = torch.optim.lr_scheduler.StepLR(
            self.optimizer_disc, step_size=decay_epoch, gamma=0.5
        )

        csv_header_needed = not os.path.exists(self.metrics_csv_path)
        os.makedirs(save_model_dir, exist_ok=True)
        for epoch in range(start_epoch, epochs):
            self.model.train()
            self.discriminator.train()
            self.maybe_refresh_lwf_snapshots(epoch)

            train_tracker = self._init_epoch_tracker()
            for idx, batch in enumerate(self.train_ds):
                step = idx + 1
                step_stats = self.train_step(batch, epoch)
                self._update_epoch_tracker(train_tracker, step_stats)

                if (step % log_interval) == 0:
                    logging.info(
                        "Epoch %d, Step %d, gen_loss=%.4f, disc_loss=%.4f, pesq=%.4f, lwf_lambda=%.4f",
                        epoch,
                        step,
                        step_stats["gen_loss"],
                        step_stats["disc_loss"],
                        step_stats["pesq_mean"] or 0.0,
                        step_stats["lwf_lambda"],
                    )

            train_stats = self._finalize_epoch_tracker(train_tracker)

            self.model.eval()
            self.discriminator.eval()

            test_tracker = self._init_epoch_tracker()
            for idx, batch in enumerate(self.test_ds):
                try:
                    step_stats = self.test_step(batch, compute_pesq=(idx == 0))
                except TypeError:
                    step_stats = self.test_step(batch)
                self._update_epoch_tracker(test_tracker, step_stats)

            test_stats = self._finalize_epoch_tracker(test_tracker)

            logging.info(
                "Epoch %d completed: train_loss=%.4f, test_loss=%.4f, test_pesq=%.4f, lwf_lambda=%.4f",
                epoch,
                train_stats["gen_loss"],
                test_stats["gen_loss"],
                test_stats["pesq_mean"] or 0.0,
                train_stats["lwf_lambda"],
            )

            save_path = os.path.join(save_model_dir, f"model_epoch_{epoch:03d}.pt")
            torch.save(self.model.state_dict(), save_path)

            with open(self.metrics_csv_path, "a", newline="") as f:
                writer = csv.DictWriter(
                    f,
                    fieldnames=[
                        "epoch",
                        "train_gen_loss",
                        "train_disc_loss",
                        "train_pesq",
                        "train_lwf_lambda",
                        "test_gen_loss",
                        "test_disc_loss",
                        "test_pesq",
                    ],
                )
                if csv_header_needed:
                    writer.writeheader()
                    csv_header_needed = False
                writer.writerow(
                    {
                        "epoch": epoch,
                        "train_gen_loss": train_stats["gen_loss"],
                        "train_disc_loss": train_stats["disc_loss"],
                        "train_pesq": train_stats["pesq_mean"] or 0.0,
                        "train_lwf_lambda": train_stats["lwf_lambda"],
                        "test_gen_loss": test_stats["gen_loss"],
                        "test_disc_loss": test_stats["disc_loss"],
                        "test_pesq": test_stats["pesq_mean"] or 0.0,
                    }
                )

            scheduler_G.step()
            scheduler_D.step()


NameError: name 'torch' is not defined

In [ ]:
train_ds, test_ds = load_data_notebook(data_dir, batch_size, num_workers, cut_len)
trainer = Trainer(train_ds, test_ds)
trainer

In [ ]:
# Run the LwF ablations one after another.
PILOT_ABLATION_ORDER = [
    "lwf_l0p5_n10",
    "lwf_l0p7_n10",
    "lwf_l0p5_n50",
    "lwf_l0p7_n50",
]

# Reduce the epoch count here for quick tests.
# epochs = 20

for preset_name in PILOT_ABLATION_ORDER:
    apply_ablation_preset(preset_name)
    train_ds, test_ds = load_data_notebook(data_dir, batch_size, num_workers, cut_len)
    trainer = Trainer(train_ds, test_ds)
    trainer.train()

Applied ablation:
  preset          : baseline_no_lwf
  lwf_enabled     : False
  lambda_lwf      : 0.0
  lambda_lwf_decay: 1.0
  lambda_start    : None
  lambda_end      : None
  n_reset         : 10
  save_model_dir  : ./saved_model_ablations/baseline_no_lwf


In [ ]:
# This cell is kept for manual single-run testing if needed.
# trainer.train()